In [1]:
!pip install -q requests beautifulsoup4 lxml pandas tqdm

In [2]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


### Configuración de las rutas principales

In [3]:
import os

BASE_DIR = "/content/drive/MyDrive/UniversidadLLM"

PORTAL_DIR = os.path.join(BASE_DIR, "portal")

RAW_DIR = os.path.join(PORTAL_DIR, "raw")
CLEAN_DIR = os.path.join(PORTAL_DIR, "clean")
JSON_DIR = os.path.join(PORTAL_DIR, "json")

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(CLEAN_DIR, exist_ok=True)
os.makedirs(JSON_DIR, exist_ok=True)

print("Proyecto:", BASE_DIR)

Proyecto: /content/drive/MyDrive/UniversidadLLM


Definición de las secciones principlaes a extraer

La idea es no hacer un scraping indiscriminado de todo el portal.

Comenzaría con estas áreas:


In [10]:
URLS = {
    "postgrado": [
        "https://upta.edu.ve/postgrado",
        "https://upta.edu.ve/postgrado/diplomados"
    ],

    "diplomados": [
        "https://upta.edu.ve/postgrado/diplomados",
        "https://upta.edu.ve/postgrado/diplomados/ia-generativa"
    ],

    "admisiones": [
        "https://upta.edu.ve/admisiones",
        "https://upta.edu.ve/admisiones/inscripcion"
    ],

    "vinculacion": [
        "https://upta.edu.ve/vinculacion",
        "https://upta.edu.ve/vinculacion/servicio-comunitario",
        "https://upta.edu.ve/vinculacion/convenios-comunidades"
    ],

    "investigacion": [
        "https://upta.edu.ve/investigacion"
    ]
}

### Función para descargar una página

In [4]:
import requests
from bs4 import BeautifulSoup
import time

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (X11; Linux x86_64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/120 Safari/537.36"
    )
}


def descargar_pagina(url):

    try:

        response = requests.get(
            url,
            headers=HEADERS,
            timeout=30
        )

        response.raise_for_status()

        return response.text

    except Exception as e:

        print(f"ERROR: {url}")
        print(e)

        return None

Se Extrae solo el contenido relevante

No queremos llevarnos:

menú
navbar
footer
botones
scripts
CSS
navegación repetida

La función siguiente hace esa extraxión

In [5]:
def extraer_contenido(html):

    soup = BeautifulSoup(html, "lxml")

    # Eliminar elementos que no son contenido
    for element in soup([
        "script",
        "style",
        "noscript",
        "svg",
        "nav",
        "footer"
    ]):

        element.decompose()

    # Intentar encontrar el contenido principal
    main = soup.find("main")

    if main is None:

        main = soup.body

    if main is None:

        return ""

    texto = main.get_text(
        separator="\n",
        strip=True
    )

    return texto

### Limpieza del contenido del portal

Ahora agregamos una limpieza específica.

In [6]:
import re


def limpiar_texto_portal(texto):

    # Normalizar saltos de línea
    texto = texto.replace("\r", "\n")

    # Eliminar espacios al inicio/final
    texto = re.sub(
        r'[ \t]+',
        ' ',
        texto
    )

    # Eliminar líneas vacías repetidas
    texto = re.sub(
        r'\n\s*\n+',
        '\n\n',
        texto
    )

    # Eliminar algunos elementos típicos de navegación
    patrones = [
        r'^Inicio$',
        r'^Volver al Inicio$',
        r'^Ver más$',
        r'^Ver programa$',
        r'^Explorar$',
        r'^Contactar Coordinación$'
    ]

    lineas = texto.splitlines()

    nuevas_lineas = []

    for linea in lineas:

        linea = linea.strip()

        if not linea:
            continue

        eliminar = False

        for patron in patrones:

            if re.match(
                patron,
                linea,
                flags=re.IGNORECASE
            ):

                eliminar = True
                break

        if not eliminar:

            nuevas_lineas.append(linea)

    texto = "\n".join(nuevas_lineas)

    return texto.strip()

### Extraer título y metadatos

In [7]:
from datetime import datetime


def obtener_metadata(html, url):

    soup = BeautifulSoup(
        html,
        "lxml"
    )

    # Título principal
    h1 = soup.find("h1")

    if h1:

        titulo = h1.get_text(
            " ",
            strip=True
        )

    else:

        titulo = soup.title.get_text(
            " ",
            strip=True
        ) if soup.title else ""

    return {
        "titulo": titulo,
        "url": url,
        "fecha_extraccion": datetime.now().strftime(
            "%Y-%m-%d"
        )
    }

Función completa de scraping
### Nueva sección
Ahora unimos todo:

In [8]:
def scrapear_url(url, categoria):

    print(f"Procesando: {url}")

    html = descargar_pagina(url)

    if html is None:

        return None

    metadata = obtener_metadata(
        html,
        url
    )

    texto = extraer_contenido(
        html
    )

    texto = limpiar_texto_portal(
        texto
    )

    resultado = {

        "id": None,

        "fuente": "portal",

        "categoria": categoria,

        "titulo": metadata["titulo"],

        "url": metadata["url"],

        "fecha_extraccion":
            metadata["fecha_extraccion"],

        "texto": texto
    }

    return resultado

### Ejecutar scraping de todas las secciones

In [11]:
from tqdm import tqdm

portal_data = []

contador = 1

for categoria, urls in URLS.items():

    print("\n")
    print("=" * 60)
    print(f"CATEGORIA: {categoria}")
    print("=" * 60)

    for url in tqdm(urls):

        resultado = scrapear_url(
            url,
            categoria
        )

        if resultado:

            resultado["id"] = (
                f"portal_"
                f"{categoria}_"
                f"{contador:04d}"
            )

            portal_data.append(
                resultado
            )

            contador += 1

        time.sleep(1)


print(
    f"\nPáginas procesadas: "
    f"{len(portal_data)}"
)



CATEGORIA: postgrado


  0%|          | 0/2 [00:00<?, ?it/s]

Procesando: https://upta.edu.ve/postgrado


 50%|█████     | 1/2 [00:01<00:01,  1.88s/it]

Procesando: https://upta.edu.ve/postgrado/diplomados


...



CATEGORIA: investigacion


  0%|          | 0/1 [00:00<?, ?it/s]

Procesando: https://upta.edu.ve/investigacion


100%|██████████| 1/1 [00:01<00:00,  1.20s/it]


Páginas procesadas: 10


### Guardar JSON

In [12]:
import json

output_json = os.path.join(
    JSON_DIR,
    "portal_upta.json"
)

with open(
    output_json,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        portal_data,
        f,
        ensure_ascii=False,
        indent=2
    )

print(
    "Archivo guardado:",
    output_json
)

Archivo guardado: /content/drive/MyDrive/UniversidadLLM/portal/json/portal_upta.json


### Guardar también TXT individual

Esto  permitirá inspeccionar manualmente qué fue extraído.

In [13]:
for item in portal_data:

    categoria = item["categoria"]

    filename = (
        item["id"] +
        ".txt"
    )

    path = os.path.join(
        CLEAN_DIR,
        categoria,
        filename
    )

    os.makedirs(
        os.path.dirname(path),
        exist_ok=True
    )

    with open(
        path,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(
            f"TÍTULO: {item['titulo']}\n"
        )

        f.write(
            f"URL: {item['url']}\n"
        )

        f.write(
            f"CATEGORÍA: {item['categoria']}\n"
        )

        f.write(
            f"FECHA: {item['fecha_extraccion']}\n"
        )

        f.write(
            "\n"
        )

        f.write(
            item["texto"]
        )

print("TXT generados.")

TXT generados.


### Crear un DataFrame para inspeccionarlo

In [14]:
import pandas as pd

df_portal = pd.DataFrame(
    portal_data
)

df_portal[
    [
        "id",
        "categoria",
        "titulo",
        "url"
    ]
]

,id,categoria,titulo,url
0,portal_postgrado_0001,postgrado,Estudios de Postgrado UPT Aragua,https://upta.edu.ve/postgrado
1,portal_postgrado_0002,postgrado,Diplomados de la UPT Aragua,https://upta.edu.ve/postgrado/diplomados
2,portal_diplomados_0003,diplomados,Diplomados de la UPT Aragua,https://upta.edu.ve/postgrado/diplomados
3,portal_diplomados_0004,diplomados,Diplomado en Inteligencia Artificial Generativa,https://upta.edu.ve/postgrado/diplomados/ia-ge...
4,portal_admisiones_0005,admisiones,Admisiones,https://upta.edu.ve/admisiones
5,portal_admisiones_0006,admisiones,Inicia tu Vida Universitaria,https://upta.edu.ve/admisiones/inscripcion
6,portal_vinculacion_0007,vinculacion,Vinculación Social,https://upta.edu.ve/vinculacion
7,portal_vinculacion_0008,vinculacion,Servicio Comunitario Estudiantil,https://upta.edu.ve/vinculacion/servicio-comun...
8,portal_vinculacion_0009,vinculacion,Convenios con Comunidades,https://upta.edu.ve/vinculacion/convenios-comu...
9,portal_investigacion_0010,investigacion,Investigación y Desarrollo,https://upta.edu.ve/investigacion


### Podemos revisar la cantidad de texto:

In [15]:
df_portal["caracteres"] = (
    df_portal["texto"]
    .str.len()
)

df_portal["palabras"] = (
    df_portal["texto"]
    .str.split()
    .str.len()
)

df_portal[
    [
        "categoria",
        "titulo",
        "palabras",
        "url"
    ]
]

,categoria,titulo,palabras,url
0,postgrado,Estudios de Postgrado UPT Aragua,286,https://upta.edu.ve/postgrado
1,postgrado,Diplomados de la UPT Aragua,455,https://upta.edu.ve/postgrado/diplomados
2,diplomados,Diplomados de la UPT Aragua,455,https://upta.edu.ve/postgrado/diplomados
3,diplomados,Diplomado en Inteligencia Artificial Generativa,349,https://upta.edu.ve/postgrado/diplomados/ia-ge...
4,admisiones,Admisiones,241,https://upta.edu.ve/admisiones
5,admisiones,Inicia tu Vida Universitaria,262,https://upta.edu.ve/admisiones/inscripcion
6,vinculacion,Vinculación Social,368,https://upta.edu.ve/vinculacion
7,vinculacion,Servicio Comunitario Estudiantil,375,https://upta.edu.ve/vinculacion/servicio-comun...
8,vinculacion,Convenios con Comunidades,401,https://upta.edu.ve/vinculacion/convenios-comu...
9,investigacion,Investigación y Desarrollo,309,https://upta.edu.ve/investigacion


### Crear un corpus único

También te recomiendo tener un archivo maestro:

In [17]:
corpus_path = os.path.join(
    PORTAL_DIR,
    "corpus_portal.json"
)

with open(
    corpus_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        portal_data,
        f,
        ensure_ascii=False,
        indent=2
    )

print(
    "Corpus:",
    corpus_path
)

Corpus: /content/drive/MyDrive/UniversidadLLM/portal/corpus_portal.json


### Crawler controlado

El proyecto no me quedaría solamente con las páginas que has indicado.

Por ejemplo, /postgrado actualmente contiene los cuatro PNFA y los diplomados, mientras que cada diplomado puede tener una página propia con objetivos, estructura, perfil, requisitos, etc.

Lo mismo sucede con admisiones: /admisiones/inscripcion tiene información mucho más específica sobre el proceso que la página general de admisiones.

El siguiente paso que te recomiendo es un crawler controlado, no simplemente una lista fija de URLs. Sin meterse, por ahora con las secciones:
/noticias
/congreso
/login
/portal-estudiante

In [18]:
from urllib.parse import urljoin, urlparse
from collections import deque


def obtener_enlaces(
    html,
    url_actual,
    prefijo
):

    soup = BeautifulSoup(
        html,
        "lxml"
    )

    enlaces = set()

    dominio_base = urlparse(
        url_actual
    ).netloc

    for a in soup.find_all(
        "a",
        href=True
    ):

        href = a["href"].strip()

        # Ignorar enlaces no HTTP
        if href.startswith(
            (
                "#",
                "mailto:",
                "tel:",
                "javascript:"
            )
        ):

            continue

        url = urljoin(
            url_actual,
            href
        )

        parsed = urlparse(url)

        # Solo dominio upta.edu.ve
        if parsed.netloc != dominio_base:

            continue

        # Normalizar
        url = url.split("#")[0]

        # Solo URLs de la sección
        path = parsed.path.rstrip("/")

        if (
            path == prefijo
            or path.startswith(
                prefijo + "/"
            )
        ):

            enlaces.add(url)

    return enlaces

### Función crawler

In [19]:
def crawler_seccion(
    url_inicio,
    categoria,
    max_paginas=100
):

    dominio = urlparse(
        url_inicio
    )

    prefijo = dominio.path.rstrip("/")

    visitadas = set()

    pendientes = deque(
        [url_inicio]
    )

    resultados = []

    while pendientes and len(
        visitadas
    ) < max_paginas:

        url = pendientes.popleft()

        if url in visitadas:

            continue

        visitadas.add(url)

        print(
            f"[{len(visitadas)}] {url}"
        )

        html = descargar_pagina(
            url
        )

        if html is None:

            continue

        # Extraer página
        metadata = obtener_metadata(
            html,
            url
        )

        texto = extraer_contenido(
            html
        )

        texto = limpiar_texto_portal(
            texto
        )

        resultados.append({

            "id": (
                f"portal_"
                f"{categoria}_"
                f"{len(resultados)+1:04d}"
            ),

            "fuente": "portal",

            "categoria": categoria,

            "titulo": metadata["titulo"],

            "url": url,

            "fecha_extraccion":
                metadata[
                    "fecha_extraccion"
                ],

            "texto": texto
        })

        # Buscar enlaces internos
        enlaces = obtener_enlaces(
            html,
            url,
            prefijo
        )

        for enlace in enlaces:

            if enlace not in visitadas:

                pendientes.append(
                    enlace
                )

        time.sleep(1)

    return resultados

Ejecución

Lo vamos a ejecutar de manera separada para poder analizar y controlar los resultados.

### Postgrado

In [20]:
postgrado = crawler_seccion(
    "https://upta.edu.ve/postgrado",
    "postgrado",
    max_paginas=100
)

print(
    "Páginas encontradas:",
    len(postgrado)
)

[1] https://upta.edu.ve/postgrado
[2] https://upta.edu.ve/postgrado/diplomados/automatizacion-industrial
[3] https://upta.edu.ve/postgrado/automatizacion
[4] https://upta.edu.ve/postgrado/diplomados/ia-generativa
[5] https://upta.edu.ve/postgrado/diplomados/mantenimiento-industrial
[6] https://upta.edu.ve/postgrado/diplomados/gerencia-mantenimiento
[7] https://upta.edu.ve/postgrado/energia-electrica
[8] https://upta.edu.ve/postgrado/diplomados
[9] https://upta.edu.ve/postgrado/diplomados/negocios-ia
[10] https://upta.edu.ve/postgrado/diplomados/gerencia-nuevas-tecnologias
[11] https://upta.edu.ve/postgrado/mecanica
[12] https://upta.edu.ve/postgrado/informatica
Páginas encontradas: 12


### Admisiones

In [21]:
admisiones = crawler_seccion(
    "https://upta.edu.ve/admisiones",
    "admisiones",
    max_paginas=100
)

[1] https://upta.edu.ve/admisiones
[2] https://upta.edu.ve/admisiones/inscripcion
[3] https://upta.edu.ve/admisiones/requisitos
[4] https://upta.edu.ve/admisiones/control-estudios


### Vinculación Social

In [22]:
vinculacion = crawler_seccion(
    "https://upta.edu.ve/vinculacion",
    "vinculacion",
    max_paginas=100
)

[1] https://upta.edu.ve/vinculacion
[2] https://upta.edu.ve/vinculacion/voluntariado
[3] https://upta.edu.ve/vinculacion/convenios-comunidades
[4] https://upta.edu.ve/vinculacion/proyectos-educativos
[5] https://upta.edu.ve/vinculacion/asesoria-tecnica
[6] https://upta.edu.ve/vinculacion/proyectos-sociotecnologicos
[7] https://upta.edu.ve/vinculacion/proyectos-productivos
[8] https://upta.edu.ve/vinculacion/servicio-comunitario
[9] https://upta.edu.ve/vinculacion/cursos-comunidad
[10] https://upta.edu.ve/vinculacion/proyectos-socio-integradores
ERROR: https://upta.edu.ve/vinculacion/proyectos-socio-integradores
404 Client Error: Not Found for url: https://upta.edu.ve/vinculacion/proyectos-socio-integradores


### Investigación

In [23]:
investigacion = crawler_seccion(
    "https://upta.edu.ve/investigacion",
    "investigacion",
    max_paginas=100
)

[1] https://upta.edu.ve/investigacion
[2] https://upta.edu.ve/investigacion/talleres
[3] https://upta.edu.ve/investigacion/centros
[4] https://upta.edu.ve/investigacion/cursos-formacion-profesional
[5] https://upta.edu.ve/investigacion/repositorio-proyectos
[6] https://upta.edu.ve/investigacion/revista-saberes-tecnologicos
[7] https://upta.edu.ve/investigacion/proyectos-socio-integradores


### Lo unimos todo

In [24]:
corpus_portal = (
    postgrado +
    admisiones +
    vinculacion +
    investigacion
)

print(
    "Total de páginas:",
    len(corpus_portal)
)

Total de páginas: 32


### Eliminamos los duplicados

In [25]:
unicos = {}

for item in corpus_portal:

    unicos[item["url"]] = item

corpus_portal = list(
    unicos.values()
)

print(
    "Páginas únicas:",
    len(corpus_portal)
)

Páginas únicas: 32


### Guardamos el corpus definitivo

In [26]:
final_path = os.path.join(
    PORTAL_DIR,
    "corpus_portal_completo.json"
)

with open(
    final_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        corpus_portal,
        f,
        ensure_ascii=False,
        indent=2
    )

print(
    f"Corpus guardado en:\n{final_path}"
)

Corpus guardado en:
/content/drive/MyDrive/UniversidadLLM/portal/corpus_portal_completo.json


## Una decisión importante para el proyecto

No vamos mezclar aún  este corpus con los PDFs.

Vamos a mantener esta estructura:
FUENTE 1
PDFs institucionales
        ↓
corpus_documentos.json


FUENTE 2
Portal upta.edu.ve
        ↓
corpus_portal.json

### Para Luego:

                 ┌── PDFs
                 │
                 ├── Portal
                 │
                 └── Datos estructurados
                        ↓
                 CORPUS INSTITUCIONAL
                        ↓
                 LIMPIEZA FINAL
                        ↓
                 CHUNKING SEMÁNTICO
                        ↓
             GENERACIÓN DE PREGUNTAS
                        ↓
                   QA DATASET
                        ↓
                    QLoRA
                        ↓
                 MODELO UPT ARAGUA

La idea además es conservar cada url, categoria, titulo y fecha_extraccion hasta el último paso. No los vamos a descartar cuando se cree el dataset QA. Esto permitirá auditar de dónde salió cada respuesta

Y hay un detalle particularmente importante: el portal contiene información que puede cambiar —por ejemplo, convocatorias 2026, diplomados, horarios, aranceles y procesos de inscripción—, así que para el modelo conviene marcar esos registros como información temporal. La página del Diplomado en IA Generativa, por ejemplo, incluye actualmente requisitos y aranceles concretos, mientras que admisiones contiene pasos operativos de inscripción.

El siguiente paso que vamos a realizar antes de generar QA es todavía más interesante: vamos a tomar este corpus_portal_completo.json + el corpus de PDFs y construir un pipeline de normalización semántica, que detecte títulos, subtítulos, listas, tablas, requisitos, pasos de procedimientos, teléfonos, correos, fechas y montos, y los convierta en fragmentos estructurados. Eso hará que los 5.000–10.000 ejemplos que generemos después sean muchísimo mejores que simplemente cortar el texto cada 2.500 caracteres.